# 4 - MLTSA on real data

**Placeholder notebook.**

For now we use the public data from the workshop repository below:

- <https://github.com/rostaresearch/enhanced-sampling-workshop-2022/tree/main/Day1/4.MLTSA/data>

These are already-calculated collective variables, so this notebook focuses on:

1. downloading the files
2. loading them cleanly
3. splitting trajectories into train, validation, and unseen test sets
4. concatenating frames within each split
5. running a small MLTSA-style analysis on top of them

This is intentionally not the final polished MD workflow notebook. It is a compact bridge
until the full end-to-end real-data tutorial is ready.

In [ ]:
from pathlib import Path
import sys

try:
    import mltsa  # noqa: F401
except ImportError:
    for parent in (Path.cwd(), *Path.cwd().parents):
        src_dir = parent / "src"
        if (src_dir / "mltsa").exists():
            sys.path.insert(0, str(src_dir))
            break
    import mltsa  # noqa: F401

import numpy as np
import matplotlib.pyplot as plt

for parent in (Path.cwd(), *Path.cwd().parents):
    notebooks_dir = parent / "notebooks"
    if notebooks_dir.exists():
        DATA_DIR = notebooks_dir / "_generated"
        break
else:
    DATA_DIR = Path.cwd() / "_generated"

DATA_DIR.mkdir(exist_ok=True, parents=True)
plt.style.use("seaborn-v0_8-whitegrid")
np.set_printoptions(precision=3, suppress=True)

## Step 1: download the placeholder workshop files

In [ ]:
import json
from urllib.request import Request, urlopen

WORKSHOP_API = (
    "https://api.github.com/repos/rostaresearch/"
    "enhanced-sampling-workshop-2022/contents/Day1/4.MLTSA/data?ref=main"
)
WORKSHOP_HEADERS = {"User-Agent": "mltsa-tutorial"}

workshop_dir = DATA_DIR / "workshop_placeholder"
workshop_dir.mkdir(exist_ok=True)


def download_workshop_data(target_dir: Path) -> list[Path]:
    request = Request(WORKSHOP_API, headers=WORKSHOP_HEADERS)
    with urlopen(request, timeout=60) as response:
        listing = json.load(response)

    downloaded: list[Path] = []
    for item in listing:
        name = item["name"]
        if not name.endswith((".csv", ".npy", ".txt")):
            continue

        destination = target_dir / name
        if not destination.exists():
            file_request = Request(item["download_url"], headers=WORKSHOP_HEADERS)
            with urlopen(file_request, timeout=120) as response:
                destination.write_bytes(response.read())
        downloaded.append(destination)

    return sorted(downloaded)


downloaded_files = download_workshop_data(workshop_dir)
for path in downloaded_files:
    print(path.name)

## Step 2: load the CVs and labels

We download everything in the folder, including:

- `allres_features.csv`
- `allres_features.npy`
- the precomputed CV arrays in `downhill_*.npy`
- the labels in `downhill_labels.txt`

For this placeholder analysis we only need the CSV names, the `downhill_*` arrays, and
the labels. The pickled `allres_features.npy` file is preserved locally as well, but we do
not depend on it here.

In [ ]:
import numpy as np

feature_name_path = workshop_dir / "allres_features.csv"
downhill_1_path = workshop_dir / "downhill_allres1.npy"
downhill_2_path = workshop_dir / "downhill_allres2.npy"
labels_path = workshop_dir / "downhill_labels.txt"
metadata_path = workshop_dir / "allres_features.npy"

feature_names = [
    line.strip()
    for line in feature_name_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
downhill_1 = np.load(downhill_1_path, allow_pickle=True)
downhill_2 = np.load(downhill_2_path, allow_pickle=True)
label_text = np.array(
    [
        line.strip()
        for line in labels_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
)

try:
    optional_metadata = np.load(metadata_path, allow_pickle=True)
    print("Optional metadata loaded:", optional_metadata.shape)
except Exception as exc:
    print("Optional metadata was downloaded but is not needed here:", type(exc).__name__)

raw_cv = np.concatenate([downhill_1, downhill_2], axis=0)
classes, y = np.unique(label_text, return_inverse=True)

print("Raw CV shape:", raw_cv.shape)
print("Class labels:", classes.tolist())
print("Feature count:", len(feature_names))

## Step 3, 4 and 5: split trajectories, concatenate frames, and run MLTSA

We keep the full feature dimension. The only slicing we do is in time: frames `100:200`.

The split is done at the **trajectory** level first. Only after that do we concatenate
frames inside each split. That way the test set still contains unseen trajectories.

In [ ]:
from sklearn.model_selection import train_test_split

from mltsa.explain import analyze, plot_importances
from mltsa.models import get_model

frame_window = slice(100, 200)  # Keep only the middle part of each trajectory
X_window = raw_cv[:, frame_window, :]

X_train_traj, X_test_traj, y_train_traj, y_test_traj = train_test_split(
    X_window,
    y,
    test_size=0.25,
    random_state=0,
    stratify=y,
)
X_train_traj, X_val_traj, y_train_traj, y_val_traj = train_test_split(
    X_train_traj,
    y_train_traj,
    test_size=0.25,
    random_state=0,
    stratify=y_train_traj,
)


def concatenate_frames(X_traj: np.ndarray, y_traj: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    n_traj, n_frames, n_features = X_traj.shape
    X_frames = X_traj.reshape(n_traj * n_frames, n_features)
    y_frames = np.repeat(y_traj, n_frames)
    return X_frames, y_frames


X_train, y_train = concatenate_frames(X_train_traj, y_train_traj)
X_val, y_val = concatenate_frames(X_val_traj, y_val_traj)
X_test, y_test = concatenate_frames(X_test_traj, y_test_traj)

print("Trajectory split shapes:")
print("  train:", X_train_traj.shape)
print("  validation:", X_val_traj.shape)
print("  test:", X_test_traj.shape)
print("Frame-concatenated shapes:")
print("  train:", X_train.shape)
print("  validation:", X_val.shape)
print("  test:", X_test.shape)

model = get_model(
    "random_forest",
    n_estimators=200,  # Forest size
    min_samples_leaf=2,  # Mild regularization
)
model.fit(X_train, y_train)
validation_accuracy = model.score(X_val, y_val)
test_accuracy = model.score(X_test, y_test)

result = analyze(
    model,
    method="native",
    feature_names=tuple(feature_names),
)

print(f"Placeholder validation accuracy: {validation_accuracy:.3f}")
print(f"Placeholder test accuracy: {test_accuracy:.3f}")
print("Top workshop features:")
for index in result.ranked_indices[:10]:
    print(f"  {result.feature_names[index]} -> {result.importances[index]:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
plot_importances(result, top_n=15, ax=ax)
plt.tight_layout()
plt.show()